# Samara five-model speller comparison (Colab)

Within-subject character benchmark on Samara P300 data: train **SVM, EEGNet, BaseCNN, ContextualTransformer, SequenceClassifier**, run the speller benchmark, and compare `char_acc(r)` curves.

Design spec: [`docs/superpowers/specs/2026-08-11-samara-colab-speller-compare-design.md`](../../docs/superpowers/specs/2026-08-11-samara-colab-speller-compare-design.md).

Absolute imports only: `from pattern_recognition...`. Requires Samara `.mat` files (no synthetic fallback).

## Colab setup (optional locally)

Run these cells on Google Colab with a GPU runtime. Skip when running from a local checkout that already has the package and data.

In [ ]:
# Install from GitHub (or pip install -e . from a Drive-mounted clone)
!pip install -q "git+https://github.com/Sidl419/pattern_recognition.git"

# Or, if the repo is already cloned / Drive-mounted:
# %cd /content/drive/MyDrive/pattern_recognition
# !pip install -q -e .

In [ ]:
# Mount Drive if Samara data / results live there
from google.colab import drive

drive.mount("/content/drive")

## Paths and settings

In [ ]:
from pathlib import Path
import json
import copy

REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "configs").is_dir() and (REPO_ROOT.parent.parent / "configs").is_dir():
    REPO_ROOT = REPO_ROOT.parent.parent

SAMARA_PATH = REPO_ROOT / "Samara_data"  # override on Colab, e.g. Drive path
NUM_EPOCHS = 250
DEVICE = "auto"
OUTPUT_DIR = REPO_ROOT / "results"

print("repo root:", REPO_ROOT)
print("Samara path:", SAMARA_PATH)

## Sanity check — Samara data required

In [ ]:
if not SAMARA_PATH.is_dir() or not any(SAMARA_PATH.glob("*.mat")):
    raise FileNotFoundError(
        f"Samara .mat files not found under {SAMARA_PATH}. "
        "Mount Drive or set SAMARA_PATH to your Samara_data directory."
    )
print(f"Found {len(list(SAMARA_PATH.glob('*.mat')))} .mat files in {SAMARA_PATH}")

## Train five models

Flash scorers (SVM, EEGNet, BaseCNN) use `SamaraWithinSubjectAverage` with `n_average=1`. ContextualTransformer and SequenceClassifier use `SamaraSelectionPackets`. Neural models train for `NUM_EPOCHS` (default 250); SVM fits in one shot.

In [ ]:
from pattern_recognition.experiment import run_experiment

NEURAL_MODELS = {"EEGNet", "BaseCNN", "ContextualTransformer", "SequenceClassifier"}

TRAIN_CONFIGS = {
    "SVM": "samara_pz_svm_sc_n1.json",
    "EEGNet": "samara_pz_eegnet_sc_n1.json",
    "BaseCNN": "samara_pz_basecnn_sc_n1.json",
    "ContextualTransformer": "samara_contextual_transformer.json",
    "SequenceClassifier": "samara_sequence_classifier.json",
}


def prepare_train_config(cfg: dict, model_name: str) -> dict:
    cfg = copy.deepcopy(cfg)
    cfg["device"] = DEVICE
    cfg["output_dir"] = str(OUTPUT_DIR)
    if "data" in cfg and "params" in cfg["data"]:
        cfg["data"]["params"]["path"] = str(SAMARA_PATH)
    if model_name in NEURAL_MODELS and "train" in cfg:
        cfg["train"]["num_epochs"] = NUM_EPOCHS
    return cfg


run_dirs = {}
for name, fname in TRAIN_CONFIGS.items():
    cfg = json.loads((REPO_ROOT / "configs" / fname).read_text())
    cfg = prepare_train_config(cfg, name)
    print(f"Training {name} ...")
    run_dirs[name] = run_experiment(cfg)
    print(f"  -> {run_dirs[name]}")

## Speller benchmark

Flash scorers use `flash_scorer`; SequenceClassifier uses `selection_classifier`. All runs share phrase `JUST_DO_IT` and repetitions `[1, 2, 5, 10]`.

In [ ]:
from pattern_recognition.speller import run_speller_benchmark

SPELLER_MODE = {
    "SVM": ("flash_scorer", "speller_samara_flash_n1.json"),
    "EEGNet": ("flash_scorer", "speller_samara_flash_n1.json"),
    "BaseCNN": ("flash_scorer", "speller_samara_flash_n1.json"),
    "ContextualTransformer": ("flash_scorer", "speller_samara_contextual.json"),
    "SequenceClassifier": ("selection_classifier", "speller_samara_sequence_classifier.json"),
}


def prepare_speller_config(cfg: dict, *, run_dir, model_mode: str, tag: str) -> dict:
    cfg = copy.deepcopy(cfg)
    cfg["run_dir"] = str(run_dir)
    cfg["model_mode"] = model_mode
    cfg["tag"] = tag
    if "protocol_params" in cfg:
        cfg["protocol_params"]["path"] = str(SAMARA_PATH)
    return cfg


speller_dirs = {}
for name, run_dir in run_dirs.items():
    mode, fname = SPELLER_MODE[name]
    cfg = json.loads((REPO_ROOT / "configs" / fname).read_text())
    cfg = prepare_speller_config(
        cfg,
        run_dir=run_dir,
        model_mode=mode,
        tag=f"compare_{name.lower()}",
    )
    print(f"Speller {name} ({mode}) ...")
    speller_dirs[name] = run_speller_benchmark(cfg)
    print(f"  -> {speller_dirs[name]}")

## Compare results

Character accuracy vs flash repetition count, plus optional binary training metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pattern_recognition.reporting import compare_speller_runs, metrics_table

rows = []
for name, speller_dir in speller_dirs.items():
    metrics_path = speller_dir / "speller_metrics.json"
    metrics = json.loads(metrics_path.read_text())
    for point in metrics["acc_vs_repeats"]:
        rows.append(
            {
                "model": name,
                "r": point["r"],
                "char_acc": point["char_acc"],
                "itr": point["itr"],
            }
        )

char_acc_table = pd.DataFrame(rows)
pivot = char_acc_table.pivot(index="r", columns="model", values="char_acc")
try:
    display(pivot)
except NameError:
    print(pivot)

fig = compare_speller_runs(list(speller_dirs.values()))
plt.show()

binary_table = metrics_table(list(run_dirs.values()))
print("\nBinary training metrics:")
try:
    display(binary_table)
except NameError:
    print(binary_table)